# Universidad Central · Maestría en Analítica de Datos
# S10 · De imágenes y audio a evidencia multimodal

**Problema de Laura.** Después de organizar noticias y documentos (S03–S08) y recuperar textos (S09), ¿cómo prepara, mide, compara y consulta **imágenes y audio** sin confundir propiedades técnicas con significado?

**Dos recursos para esta clase:** [Presentación-aplicación y guía integrada](https://jazaineam1.github.io/BigData2026/Presentaciones/s10-imagenes-audio.html#s1) y **este cuaderno**. No requiere cuentas de pago, credenciales de terceros ni descarga de audios periodísticos.

**Qué vas a producir:** originales propios del laboratorio, dos tamaños de imagen, una señal WAV, histogramas, formas de onda y espectrograma, características tabulares, catálogo JSONL/Parquet, manifiesto y decisión individual cerrada.

**Mapa sin relojes:** caso y límites → imagen: píxeles → imagen: transformación → audio: muestras → audio: ventanas y espectro → catálogo → consulta → decisión y conservación.

**Derechos y límites:** las direcciones de audio indicadas por feeds editoriales se estudian como metadatos, no se rastrean ni se descargan. El laboratorio usa una ilustración y una señal generadas localmente. Si analizas tu propia grabación, pide permiso cuando intervengan otras personas; en Colab, los archivos que subes llegan al entorno de Google.

## 1 · Preparar un entorno que pueda repetirse

**HAZ ESTO AHORA.** Ejecuta las celdas en orden. En Colab guarda una copia de tu cuaderno en Drive. El ejemplo completo no requiere subir archivos: crea sus propias entradas.

**Funciones:** \`Path.mkdir()\` crea carpetas, \`importlib.util.find_spec()\` comprueba bibliotecas y \`subprocess.check_call()\` instala solo lo ausente. Si trabajas en VS Code, usa Python 3.10 o superior.

In [ ]:
#@title Preparar bibliotecas (no reemplazar NumPy/Pandas precargados)
import importlib.util, subprocess, sys, os
faltan=[]
for modulo, paquete in [('PIL','pillow'),('numpy','numpy'),('pandas','pandas'),('pyarrow','pyarrow'),('matplotlib','matplotlib'),('IPython','ipython')]:
    if importlib.util.find_spec(modulo) is None: faltan.append(paquete)
if faltan: subprocess.check_call([sys.executable,'-m','pip','install','-q',*faltan])
print('BIBLIOTECAS:', 'listas; no se necesitan servicios externos')
from IPython.display import display, Audio
from pathlib import Path
RAIZ=Path('work_s10_multimodal').resolve()
RAW=RAIZ/'raw'; DER=RAIZ/'derived'; OUT=RAIZ/'outputs'; CHECKS=RAIZ/'checks'
for carpeta in (RAW,DER,OUT,CHECKS): carpeta.mkdir(parents=True,exist_ok=True)
print('ÁREA DE TRABAJO:',RAIZ)

## 2 · Contexto: la misma clave no es el mismo contenido

**Ejemplo de esquema proporcionado para la clase:** una noticia puede incluir \`audio_url\`; otras referencias apuntan a imágenes. **Aquí solo reconocemos y validamos campos**, no solicitamos esos archivos ni concluimos que imagen y audio contienen lo mismo.

**Cómo se lee:** \`article_id\` identifica el registro editorial, \`audio_url\` es una referencia declarada. **Qué nos dice:** cómo unir metadatos de distintas modalidades. **Qué NO permite concluir:** que el audio existe ahora, que tiene licencia para minería o que describe la foto. **Error común:** inferir disponibilidad o derechos a partir de una URL.

In [ ]:
# Ejemplo de estructura, sin consultar ningún sitio periodístico.
from urllib.parse import urlparse
feed_ejemplo={'article_id':'3530042',
             'article_feed':'https://www.eltiempo.com/servicios/feeds/articulo/3530042',
             'audio_url':'https://servicios.eltiempo.com/audios/3530042.wav'}
assert urlparse(feed_ejemplo['audio_url']).scheme=='https'
assert feed_ejemplo['audio_url'].endswith('/'+feed_ejemplo['article_id']+'.wav')
print('REFERENCIA DE AUDIO (NO DESCARGADA):',feed_ejemplo['audio_url'])
print('ACTIVOS DEL LABORATORIO: originales del curso, independientes de ese artículo')

## 3 · Imagen: de una matriz a preguntas observables

Una imagen RGB guarda tres canales por píxel. Su resolución (ancho × alto) no es el número de bytes comprimidos ni indica, por sí sola, legibilidad.

**Ejemplo manual:** 4 × 3 píxeles son 12 posiciones. Si cada una tiene tres canales de 8 bits, la matriz sin comprimir ocupa 36 bytes, sin contar metadatos. Un PNG o un JPEG almacenado puede ocupar otra cantidad.

**HAZ ESTO AHORA.** Genera una imagen que represente un documento de prueba; no se copia ningún documento real.

In [ ]:
# E — imagen ORIGINAL creada por el curso (sin fuentes de terceros).
from PIL import Image,ImageDraw,ImageFont
import numpy as np, hashlib, json, shutil, wave, math
import pandas as pd, matplotlib.pyplot as plt
def sha256(ruta):
    h=hashlib.sha256()
    with open(ruta,'rb') as f:
        for bloque in iter(lambda:f.read(1024*1024),b''):h.update(bloque)
    return h.hexdigest()
imagen_original=RAW/'documento_demo.png'
if not imagen_original.exists():
    im=Image.new('RGB',(640,360),'#e8edf6');d=ImageDraw.Draw(im)
    d.rectangle((38,22,599,337),fill='white',outline='#1c365b',width=6)
    d.rectangle((70,52,310,94),fill='#23647a')
    for n in range(11):d.rectangle((72,116+n*17,405+(n%3)*42,121+n*17),fill='#657a90')
    d.rectangle((440,68,550,192),fill='#ebc46c')
    d.ellipse((460,93,528,164),fill='#4b9091')
    d.text((70,305),'BIG DATA - DOCUMENTO FICTICIO',fill='#213b52')
    im.save(imagen_original)
print('E · ORIGINAL:',imagen_original.name,sha256(imagen_original)[:16])
display(Image.open(imagen_original))

### Pregunta: ¿reducir una imagen conserva toda la información?

**HAZ ESTO AHORA.** Cambia **solo** \`LADO=320\` por \`160\` y vuelve a ejecutar. Compara proporción, bytes y legibilidad. Luego decide cuál perfil sirve como vista previa del catálogo.

**Función:** \`Image.thumbnail((lado,lado))\` reduce manteniendo proporción. No modifica el archivo RAW si trabajas con una copia y escribes un derivado distinto.

In [ ]:
# T — un solo parámetro: 160 o 320. El RAW permanece intacto.
LADO=320 #@param [160,320] {type:"raw"}
if type(LADO) is not int or LADO not in (160,320): raise ValueError('Elige 160 o 320')
hash_antes=sha256(imagen_original)
derivada=DER/f'documento_{LADO}.png'
with Image.open(imagen_original) as im:
    copia=im.copy();copia.thumbnail((LADO,LADO),Image.Resampling.LANCZOS);copia.save(derivada)
assert sha256(imagen_original)==hash_antes
with Image.open(derivada) as im:
    print('T · TAMAÑO:',im.size,'· bytes:',derivada.stat().st_size,'· RAW conservado:',True)
    display(im)

### De una imagen a números

**Función:** \`numpy.asarray()\` transforma la imagen en una matriz. El promedio de luminancia describe brillo general; el histograma cuenta intensidades. Una diferencia entre píxeles vecinos aproxima cambios locales, pero no reconoce por sí sola palabras, personas u objetos.

Observa las tres salidas: matriz reducida, histograma y proporción de cambios locales.

In [ ]:
# Propiedades cuantificables del mismo original y los dos perfiles.
def imagen_caracteristicas(ruta):
    with Image.open(ruta) as img:
        rgb=np.asarray(img.convert('RGB'),dtype=np.float32)
    gris=.2126*rgb[:,:,0]+.7152*rgb[:,:,1]+.0722*rgb[:,:,2]
    h,w=gris.shape
    dx=np.abs(np.diff(gris,axis=1));dy=np.abs(np.diff(gris,axis=0))
    return {'width':int(w),'height':int(h),'pixels':int(w*h),
      'mean_luminance':round(float(gris.mean()),3),
      'bright_fraction':round(float((gris>=180).mean()),4),
      'horizontal_change':round(float(dx.mean()),3),
      'vertical_change':round(float(dy.mean()),3),
      'stored_bytes':int(Path(ruta).stat().st_size)}
comparacion=[]
for lado in (160,320):
    ruta=DER/f'documento_{lado}.png'
    with Image.open(imagen_original) as im:
        copia=im.copy();copia.thumbnail((lado,lado),Image.Resampling.LANCZOS);copia.save(ruta)
    comparacion.append({'perfil':f'vista{lado}',**imagen_caracteristicas(ruta)})
display(pd.DataFrame(comparacion))
grises=np.asarray(Image.open(imagen_original).convert('L'))
plt.figure(figsize=(8,2.7));plt.hist(grises.ravel(),bins=32,range=(0,256));plt.xlabel('Intensidad (0–255)');plt.ylabel('Píxeles');plt.title('Histograma del original del curso');plt.show()
print('CÓMO SE LEE: compara el cambio de resolución y de distribución; el histograma no devuelve texto OCR.')

**Cómo se lee:** cada fila corresponde a un tamaño y su brillo, sus cambios locales y bytes almacenados. **Qué nos dice:** el efecto de reducir esta ilustración particular. **Qué NO permite concluir todavía:** calidad de OCR en documentos reales; necesitaríamos un conjunto de documentos y un texto de referencia. **Qué error común:** igualar tamaño comprimido con cantidad de información visual útil.

**Fállalo aquí, que sale gratis:** ¿por qué un histograma no identifica el artículo de una foto? Porque cuenta intensidades sin conservar todas sus ubicaciones.

## 4 · Audio: la señal antes de las palabras

Un archivo WAV puede contener muestras PCM. A \`16000\` muestras por segundo, una señal de tres segundos tiene \`48000\` valores por canal. La frecuencia de muestreo no es la altura del tono.

**HAZ ESTO AHORA.** Genera una señal ORIGINAL didáctica. Tiene dos tonos, un silencio y amplitudes distintas; **no es voz ni una noticia**. La emplearemos para reconocer ventanas temporales, RMS, frecuencias dominantes y límites interpretativos.

In [ ]:
# E — audio WAV PCM mono de 16 bits, creado localmente sin terceros.
SR=16000
duracion=3.0
t=np.arange(int(SR*duracion),dtype=np.float64)/SR
senal=np.zeros(t.size,dtype=np.float64)
primero=(t>=.1)&(t<1.0);segundo=(t>=2.0)&(t<2.9)
senal[primero]=.45*np.sin(2*np.pi*190*t[primero])+.09*np.sin(2*np.pi*380*t[primero])
senal[segundo]=.30*np.sin(2*np.pi*360*t[segundo])+.04*np.sin(2*np.pi*720*t[segundo])
audio_original=RAW/'senal_demo.wav'
pcm=(np.clip(senal,-1,1)*32767).astype('<i2')
with wave.open(str(audio_original),'wb') as wf:
    wf.setnchannels(1);wf.setsampwidth(2);wf.setframerate(SR);wf.writeframes(pcm.tobytes())
assert audio_original.exists()
from IPython.display import Audio,display
display(Audio(senal,rate=SR))
print('E · AUDIO:',round(len(senal)/SR,2),'s ·',SR,'Hz · canal mono · WAV PCM16 ·',audio_original.stat().st_size,'bytes')

## 5 · Escuchar, dibujar y medir

**RMS** (raíz del promedio de cuadrados) resume amplitud/energía; no demuestra volumen percibido ni emoción. Un espectro FFT mide las frecuencias en **una ventana**. Un espectrograma repite ese cálculo a lo largo del tiempo.

**HAZ ESTO AHORA.** Cambia únicamente \`VENTANA_SEG=0.2\` por \`2.2\`; lee las métricas de esa ventana y compáralas con la señal completa. Después prueba \`1.3\` para observar silencio.

**Función:** \`np.fft.rfft()\` calcula componentes de frecuencia de muestras reales; \`np.fft.rfftfreq()\` devuelve frecuencias en hercios.

In [ ]:
# T — características acústicas por ventana; el audio WAV no se sobrescribe.
VENTANA_SEG=0.2 #@param [0.2,1.3,2.2] {type:"raw"}
if float(VENTANA_SEG) not in (.2,1.3,2.2):raise ValueError('Elige una ventana propuesta')
def audio_ventana(x,sample_rate,segundos,ancho=.25):
    inicio=int(segundos*sample_rate);n=int(ancho*sample_rate)
    ventana=np.asarray(x[inicio:inicio+n],dtype=np.float64)
    if len(ventana)!=n:raise ValueError('Ventana fuera de la señal')
    rms=float(np.sqrt(np.mean(ventana**2)))
    magnitud=np.abs(np.fft.rfft(ventana*np.hanning(len(ventana))))
    frecuencias=np.fft.rfftfreq(len(ventana),d=1/sample_rate)
    pico=float(frecuencias[np.argmax(magnitud[1:])+1]) if len(magnitud)>1 else 0.
    return {'start_seconds':segundos,'duration_seconds':ancho,'rms':round(rms,5),
            'peak_frequency_hz':round(pico,1),'peak_amplitude':round(float(np.max(np.abs(ventana))),4),
            'signal':ventana,'frequencies':frecuencias,'magnitudes':magnitud}
actual=audio_ventana(senal,SR,float(VENTANA_SEG))
print('VENTANA:',{k:v for k,v in actual.items() if k not in ('signal','frequencies','magnitudes')})
plt.figure(figsize=(9,2.5));plt.plot(np.arange(len(senal))/SR,senal,lw=.7);plt.xlabel('Segundos');plt.ylabel('Amplitud relativa');plt.title('Señal original: tonos y silencio');plt.show()
plt.figure(figsize=(9,2.5));plt.plot(actual['frequencies'],actual['magnitudes']);plt.xlim(0,1000);plt.xlabel('Hz');plt.ylabel('Magnitud relativa');plt.title('Espectro de TU ventana');plt.show()

### El espectrograma: frecuencias que cambian con el tiempo

Un espectrograma hace ventanas consecutivas. En este caso **debe** haber una zona prácticamente silenciosa, además de dos regiones tonales. Las frecuencias del ejemplo son conocidas por diseño; eso nos permite comprobar nuestro cálculo.

Un espectrograma no revela automáticamente quién habla ni las palabras de una noticia.

In [ ]:
# Espectrograma: FFT por segmentos, con salto menor que el tamaño de ventana.
ancho=512;salto=160
bloques=np.stack([senal[i:i+ancho] for i in range(0,len(senal)-ancho+1,salto)])
espectros=np.abs(np.fft.rfft(bloques*np.hanning(ancho),axis=1))
frecuencias=np.fft.rfftfreq(ancho,1/SR)
instantes=(np.arange(len(bloques))*salto+ancho/2)/SR
plt.figure(figsize=(9,3))
plt.pcolormesh(instantes,frecuencias,20*np.log10(espectros.T+1e-6),shading='auto')
plt.ylim(0,1200);plt.xlabel('Tiempo (s)');plt.ylabel('Frecuencia (Hz)')
plt.title('Espectrograma del audio original; dB relativos');plt.colorbar(label='dB relativos');plt.show()
metricas_ventanas=pd.DataFrame([{k:v for k,v in audio_ventana(senal,SR,s).items() if k not in ('signal','frequencies','magnitudes')} for s in (.2,1.3,2.2)])
display(metricas_ventanas)

**Cómo se lee:** hay frecuencias distintas en las regiones sonoras y un valle de energía en silencio. **Qué nos dice:** cómo cambió esta señal conocida. **Qué NO permite concluir:** que existe una voz o un contenido semántico. **Error común:** confundir muestreo (Hz del dispositivo) con pico de tono (Hz de la señal).

### Si tienes un WAV propio con autorización (opcional)
Puedes sustituir la señal de ejemplo con tu voz grabada voluntariamente, o un audio con licencia de análisis. Usa WAV PCM mono para seguir el recorrido sin instalar un decodificador adicional. **No subas conversaciones privadas o sin consentimiento a Colab.** Si cambias de archivo, reinicia los cálculos; no le atribuyas al audio del curso el contenido de tu grabación.

In [ ]:
#@title Opcional: usar WAV PCM propio (DESACTIVADO para ejecución de clase)
USAR_WAV_PROPIO=False #@param {type:"boolean"}
RUTA_WAV_PROPIO='' #@param {type:"string"}
AUTORIZADO_PARA_ANALISIS=False #@param {type:"boolean"}
if USAR_WAV_PROPIO:
    if not AUTORIZADO_PARA_ANALISIS:raise ValueError('Se requiere permiso para analizar este audio')
    ruta=Path(RUTA_WAV_PROPIO)
    if not ruta.is_file() or ruta.stat().st_size>20*1024*1024:raise ValueError('WAV ausente o mayor de 20 MiB')
    with wave.open(str(ruta),'rb') as wf:
        if wf.getnchannels()!=1 or wf.getsampwidth()!=2 or wf.getcomptype()!='NONE':
            raise ValueError('Usa WAV PCM16 mono, sin compresión')
        SR_PROPIO=wf.getframerate();num=min(wf.getnframes(),SR_PROPIO*30)
        muestras=np.frombuffer(wf.readframes(num),dtype='<i2').astype(np.float32)/32768
    if SR_PROPIO<8000 or SR_PROPIO>48000:raise ValueError('Muestreo fuera del alcance')
    print('AUDIO AUTORIZADO:',len(muestras)/SR_PROPIO,'segundos; RMS:',float(np.sqrt(np.mean(muestras**2))))
else:
    print('Usando la señal original del curso; sin archivos de terceros')

## 6 · Voz y significado: por qué ASR es una etapa distinta

Reconocimiento automático del habla (**ASR**) produce una hipótesis de palabras a partir de una señal que efectivamente contenga voz. Ni una FFT ni RMS transcriben palabras.

Para evaluar un sistema de transcripción necesitas una **referencia humana**; WER cuenta sustituciones, inserciones y eliminaciones de palabras y divide por la longitud de esa referencia. Con cero palabras de referencia, WER no está definido.

**Extensión de profundización (no necesaria en esta clase):** ejecutar un modelo ASR descargado y autorizado en un entorno apropiado, comparar transcripción contra referencia, y observar fallos con ruido/nombres propios. No declares calidad del modelo con el audio sintético sin voz de este cuaderno.

In [ ]:
# Ejercicio manual de evaluación de ASR: ejemplos creados para el curso, no son transcripciones editoriales.
referencia='el contrato tiene tres anexos'.split()
hipotesis='el contrato tiene dos anexos'.split()
def distancia_palabras(a,b):
    anterior=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        actual=[i]+[0]*len(b)
        for j,y in enumerate(b,1):
            actual[j]=min(anterior[j]+1,actual[j-1]+1,anterior[j-1]+(x!=y))
        anterior=actual
    return anterior[-1]
errores=distancia_palabras(referencia,hipotesis)
wer=errores/len(referencia)
print('WER DEL EJEMPLO DIDÁCTICO:',errores,'errores /',len(referencia),'palabras =',round(wer,3))
assert errores==1
print('LÍMITE: WER mide comparación textual con referencia; no la verdad del contenido.')

### Laboratorio opcional · Transcribir tu propio audio con un modelo gratuito

Hasta ahora medimos sonido sin inferir palabras. Si tienes **un WAV PCM de voz propia, autorizado** y decides explorar ASR, puedes ejecutar un modelo *tiny* de Whisper usando \`faster-whisper\` en Colab **sin una API de pago**. Esta parte descarga un modelo externo solo cuando eliges ejecutarla; el tiempo y la disponibilidad dependen de Colab y del servicio que aloja el modelo.

**HAZ ESTO AHORA (solo profundización).** Graba voluntariamente la frase ficticia «El contrato tiene tres anexos» y guarda el WAV. Activa la opción, indica su ruta y revisa los errores frente a la frase conocida. No uses los audios de El Tiempo sin permiso escrito para análisis.

**Cómo se lee:** ASR produce texto probable y marcas temporales. **Qué nos dice:** qué palabras reconoció este modelo en tu grabación. **Qué NO permite concluir:** precisión en audios periodísticos ni verdad del enunciado. **Error común:** confundir la transcripción probable con una cita verificada. Los modelos pueden inventar palabras durante silencios.

In [ ]:
#@title ASR opcional sobre TU WAV autorizado (desactivado por defecto)
USAR_ASR=False #@param {type:"boolean"}
if USAR_ASR:
    if not USAR_WAV_PROPIO or not AUTORIZADO_PARA_ANALISIS:
        raise ValueError('Primero habilita un WAV de voz propia con autorización, no un archivo editorial')
    ruta=Path(RUTA_WAV_PROPIO)
    if not ruta.exists():raise FileNotFoundError('Sube tu WAV a Colab e indica la ruta')
    import importlib.util,subprocess,sys
    if importlib.util.find_spec('faster_whisper') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','faster-whisper>=1.0,<2'])
    from faster_whisper import WhisperModel
    modelo=WhisperModel('tiny',device='cpu',compute_type='int8')
    segmentos,info=modelo.transcribe(str(ruta),language='es',beam_size=2,vad_filter=True)
    segmentos=list(segmentos)
    texto_reconocido=' '.join(s.text.strip() for s in segmentos).strip()
    esperado='el contrato tiene tres anexos'.split()
    observado=texto_reconocido.lower().split()
    wer_personal=distancia_palabras(esperado,observado)/len(esperado)
    print('MODELO: tiny · idioma forzado: español · transcripción:',texto_reconocido)
    print('WER respecto de la frase de prueba:',round(wer_personal,3))
    print('No extrapolar a otras voces, idiomas o entornos.')
else:
    print('ASR opcional no ejecutado. El laboratorio obligatorio no depende de descargar un modelo.')

## 7 · Diez grabaciones **reales de voz** en español

Acabamos de comprender una señal sencilla. Ahora analizaremos **10 grabaciones reales de personas hablando español**, del conjunto **MINDS-14 / PolyAI, configuración \`es-ES\`**, publicado como **CC BY 4.0**. Son consultas habladas sobre servicios financieros; **NO** son noticias de El Tiempo, **NO** pertenecen a los 987 artículos de la colección editorial y **NO** deben unirse falsamente a ellos.

**¿Por qué este segundo origen?** Queremos ejecutar análisis acústico y transcripción de voz real sin infringir licencias. El feed de una noticia puede declarar \`audio_url\`, pero la presencia de una URL no autoriza descargar ni hacer minería de medios editoriales. El material del curso mantiene su esquema \`article_id\` para noticias, mientras los audios de MINDS-14 mantienen su propio identificador \`source_record_id\`.

**Qué tendrás al terminar:** diez WAV verdaderos en \`raw/voz_real/\`, diez huellas SHA-256, diez transcripciones de referencia, duración, RMS, cambios de signo y centroide espectral; después tendrás once filas de audio (10 reales + 1 señal de prueba) y dos filas de imagen en el catálogo.

**HAZ ESTO AHORA.** Ejecuta la próxima celda. Descarga el **Parquet de español (~39 MB) una sola vez** al entorno de Colab, sin autenticación y sin API de pago. Después selecciona de manera determinista una muestra de **diez grabaciones de intenciones diferentes**. No es una muestra aleatoria representativa. Se conservan el dataset, la licencia, el registro, los bytes y la transcripción proporcionada por el corpus.

**Si falla:** revisa conectividad de Colab y reejecuta esta celda; si el proveedor no responde se detiene con error explícito. No se sustituyen las diez voces por señales inventadas. El tiempo depende de la red.

In [ ]:
#@title 7A · Conseguir 10 grabaciones reales (MINDS-14 español, CC BY 4.0)
# Los audios son voz humana del dataset; NO descargas de sitios periodísticos.
import urllib.request, io, time, collections
ARCHIVO_FUENTE=RAIZ/'cache'/'minds14_es_ES_train.parquet'
ARCHIVO_FUENTE.parent.mkdir(parents=True,exist_ok=True)
ORIGEN_VOZ='https://huggingface.co/datasets/PolyAI/minds14/resolve/main/es-ES/train-00000-of-00001.parquet'
LICENCIA_VOZ='CC-BY-4.0'
if not ARCHIVO_FUENTE.exists():
    temporal=ARCHIVO_FUENTE.with_suffix('.parquet.part')
    print('Preparando el corpus de voz real en español (~39 MB)...')
    try:
        req=urllib.request.Request(ORIGEN_VOZ,headers={'User-Agent':'BigData-UCentral-S10-educational/1.0'})
        with urllib.request.urlopen(req,timeout=120) as respuesta, temporal.open('wb') as destino:
            while True:
                parte=respuesta.read(1024*1024)
                if not parte: break
                destino.write(parte)
                if destino.tell()>100_000_000: raise ValueError('Descarga superior al límite seguro de 100 MB')
        temporal.replace(ARCHIVO_FUENTE)
    except Exception as error:
        temporal.unlink(missing_ok=True)
        raise RuntimeError('No se pudo obtener el corpus autorizado; verifica la red de Colab y reintenta (no se usarán voces sintéticas).') from error
# No se descargan los archivos wav uno a uno: ya están encapsulados en el parquet.
corpus_voz=pd.read_parquet(ARCHIVO_FUENTE,columns=['audio','path','transcription','intent_class'])
if len(corpus_voz)<10:raise ValueError('El corpus tiene menos de 10 grabaciones')
corpus_voz=corpus_voz.sort_values('path',kind='stable').reset_index(drop=True)
escogidos=[];intenciones=set()
for i,fila in corpus_voz.iterrows():
    intento=int(fila['intent_class'])
    if intento not in intenciones:
        escogidos.append((i,fila));intenciones.add(intento)
    if len(escogidos)==10:break
if len(escogidos)<10:raise ValueError('No hay 10 categorías distintas en este corpus')
VOZ_RAW=RAW/'voz_real';VOZ_RAW.mkdir(parents=True,exist_ok=True)
audios_reales=[];hashes=set()
for indice,(fila_numero,fila) in enumerate(escogidos,1):
    contenido=fila['audio']
    if not isinstance(contenido,dict) or not isinstance(contenido.get('bytes'),(bytes,bytearray)):
        raise ValueError('Audio no está embebido como bytes en el Parquet; verificar esquema en origen')
    bytes_original=bytes(contenido['bytes'])
    if not bytes_original.startswith(b'RIFF') or bytes_original[8:12]!=b'WAVE':
        raise ValueError('El registro de audio no contiene WAV válido')
    ruta=VOZ_RAW/f'voz_{indice:02}.wav'
    # Repetir no altera los bytes y evita aceptar archivos corruptos.
    huella=hashlib.sha256(bytes_original).hexdigest()
    if huella in hashes:raise ValueError('Se detectó un archivo de audio repetido')
    hashes.add(huella)
    if ruta.exists() and sha256(ruta)!=huella: raise ValueError('El WAV guardado anteriormente tiene otra huella')
    if not ruta.exists():ruta.write_bytes(bytes_original)
    with wave.open(str(ruta),'rb') as wav:
        if wav.getnframes()<wav.getframerate()//2:raise ValueError('Grabación demasiado corta')
        if wav.getsampwidth()!=2 or wav.getcomptype()!='NONE':
            raise ValueError('Se requiere WAV PCM de 16 bits para este laboratorio')
        duracion_real=wav.getnframes()/wav.getframerate()
    audios_reales.append({'source_record_id':str(fila['path']),'source_row':int(fila_numero),
        'intent_class':int(fila['intent_class']),'transcription_reference':str(fila['transcription']),
        'path':ruta.relative_to(RAIZ).as_posix(),'sha256':huella,
        'duration_seconds':round(duracion_real,3),'license':LICENCIA_VOZ})
assert len(audios_reales)==10 and len(hashes)==10
(VOZ_RAW/'dataset_manifest.json').write_text(json.dumps({
    'dataset':'PolyAI/minds14','subset':'es-ES','license':LICENCIA_VOZ,
    'source_url':ORIGEN_VOZ,'source_parquet_sha256':sha256(ARCHIVO_FUENTE),
    'selection':'one_per_distinct_intent_sorted_by_path','records':audios_reales
},ensure_ascii=False,indent=2),encoding='utf-8')
print('VOCES REALES VERIFICADAS:',len(audios_reales),'· intenciones:',len(intenciones))
display(pd.DataFrame(audios_reales)[['source_record_id','duration_seconds','intent_class','sha256']])

### 7B · ¿Qué dicen las señales de diez personas?

**La pregunta.** Dos audios pueden tener duraciones diferentes e igual contenido temático; y dos frases con intención parecida pueden tener firmas acústicas distintas.

**Ejemplo manual:** para cuatro muestras \`[0, 1, 0, -1]\`, el RMS es \(\sqrt{(0+1+0+1)/4}\), aproximadamente 0,707. La energía depende de la forma de onda, no de si la persona menciona un contrato.

**HAZ ESTO AHORA.** Ejecuta la celda, mira las diez filas y reproduce el audio de la posición elegida (de 1 a 10). Cambia solo \`POSICION_VOZ\` y compara su duración, su RMS y su centroide espectral; contrasta con la transcripción de referencia.

**Qué significa cada característica:**
- **RMS:** amplitud eficaz; no equivale a volumen percibido ni emoción.
- **ZCR:** frecuencia de cruces por cero por muestra; puede aumentar con componentes de alta frecuencia o ruido, pero no es una clasificación de voz.
- **Centroide espectral:** media de frecuencias ponderada por magnitud espectral; no interpreta palabras.
- **Transcripción de referencia:** texto asociado por el dataset, no predicción de nuestro modelo.

**Cómo se lee:** una fila es una grabación única con procedencia. **Qué nos dice:** variación acústica medible. **Qué NO permite concluir:** la intención del hablante a partir de RMS ni la exactitud de ASR sin ejecutarlo. **Error común:** usar la clave \`article_id\` de noticias para estas voces, que no pertenecen a ellas.

In [ ]:
#@title 7B · Extraer características y escuchar una voz del corpus
def leer_wav_pcm16(ruta):
    with wave.open(str(ruta),'rb') as f:
        tasa=f.getframerate();canales=f.getnchannels();n=f.getnframes()
        if f.getcomptype()!='NONE' or f.getsampwidth()!=2 or not 1<=canales<=2:
            raise ValueError('Se espera WAV PCM de 16 bits mono o estéreo')
        muestras=np.frombuffer(f.readframes(n),dtype='<i2').astype(np.float32)/32768
    if canales>1:muestras=muestras.reshape(-1,canales).mean(axis=1)
    return muestras,tasa
def caracteristicas_voz(ruta):
    muestras,tasa=leer_wav_pcm16(ruta)
    if muestras.size<1024:raise ValueError('Archivo con muy pocas muestras')
    rms=float(np.sqrt(np.mean(muestras.astype(np.float64)**2)))
    cruces=float(np.mean(np.signbit(muestras[1:])!=np.signbit(muestras[:-1])))
    # Primeros 0,5 segundos; Hanning + FFT para el mismo contrato en todas las voces.
    ventana=muestras[:min(len(muestras),int(tasa*.5))]
    amplitudes=np.abs(np.fft.rfft(ventana*np.hanning(len(ventana))))
    hz=np.fft.rfftfreq(len(ventana),d=1/tasa)
    centroide=float(np.sum(hz*amplitudes)/max(1e-12,float(amplitudes.sum())))
    return {'duration_seconds':round(len(muestras)/tasa,3),'sample_rate_hz':int(tasa),
            'rms':round(rms,5),'zero_crossing_rate':round(cruces,5),
            'spectral_centroid_hz':round(centroide,1),'stored_bytes':int(Path(ruta).stat().st_size)}
for item in audios_reales:
    item.update(caracteristicas_voz(RAIZ/item['path']))
tabla_voz=pd.DataFrame(audios_reales)
assert len(tabla_voz)==10 and tabla_voz.sha256.nunique()==10
print('ANÁLISIS ACÚSTICO REAL:',len(tabla_voz),'grabaciones · licencia:',LICENCIA_VOZ)
display(tabla_voz[['source_record_id','intent_class','duration_seconds','rms','zero_crossing_rate','spectral_centroid_hz']])
POSICION_VOZ=1 #@param {type:"integer"}
if type(POSICION_VOZ) is not int or not 1<=POSICION_VOZ<=10:raise ValueError('Selecciona una voz de 1 a 10')
voz_elegida=audios_reales[POSICION_VOZ-1]
senal_escuchada,tasa_escuchada=leer_wav_pcm16(RAIZ/voz_elegida['path'])
print('TRANSCRIPCIÓN HUMANA DE REFERENCIA:',voz_elegida['transcription_reference'])
display(Audio(senal_escuchada,rate=tasa_escuchada))
plt.figure(figsize=(9,2));plt.plot(np.arange(len(senal_escuchada))/tasa_escuchada,senal_escuchada,lw=.65)
plt.title('Forma de onda · voz real '+str(POSICION_VOZ));plt.xlabel('Tiempo (s)');plt.ylabel('Amplitud');plt.show()

### 7C · Lo que proviene de cada fuente y cómo se relaciona con las imágenes

La sesión anterior de imágenes ya dejó un snapshot de **32 imágenes de 24 noticias** (\`Datos/s10_imagenes_reales/inputs.json\`) y un cuaderno específico (\`10_ETL_Multimedia.ipynb\`). Conservamos ese recurso y **reutilizamos su esquema de procedencia**; las diez voces reales provienen de otro conjunto, con otra licencia y otros identificadores.

**HAZ ESTO AHORA.** Lee el manifiesto de las imágenes desde el repositorio (solo metadatos) y comprueba las cifras antes de unir ambos **inventarios de medios**, no sus significados. NO descargamos fotografías del diario ni las volvemos a procesar sin la autorización pertinente.

**Salida esperada:** 32 referencias de imágenes del snapshot anterior; 10 registros de audio de MINDS-14. Se registra explícitamente si existe alguna relación real entre conjuntos: en este caso **ninguna**.

**Límite:** compartir un catálogo técnico no significa compartir un artículo. Si se recibe autorización para analizar audios del mismo medio, se deberá comprobar cada referencia, permisos y enlaces antes de cruzar por \`article_id\`.

In [ ]:
#@title 7C · Integración de inventarios sin inventar relaciones editoriales
RUTA_IMAGENES_CURSO='https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s10_imagenes_reales/inputs.json'
try:
    with urllib.request.urlopen(urllib.request.Request(RUTA_IMAGENES_CURSO,headers={'User-Agent':'BigData-UCentral-S10/1.0'}),timeout=35) as resp:
        imagenes_existentes=json.load(resp)
except Exception as e:
    raise RuntimeError('No se pudieron recuperar los metadatos del snapshot de imágenes; revisa la conexión de Colab') from e
if len(imagenes_existentes)!=32:raise ValueError('El snapshot de imágenes cambió: hay que revisar su contrato')
articulos_imagenes={str(x['article_id']) for x in imagenes_existentes}
if len(articulos_imagenes)!=24:raise ValueError('Cambiaron los article_id del inventario anterior')
registro_fuentes={'sources':[
    {'dataset':'S10 snapshot editorial previo','modality':'image','media_items':32,
     'distinct_articles':24,'id_namespace':'article_id','content_analysis':'sin nueva extracción'},
    {'dataset':'PolyAI/minds14 es-ES','modality':'audio','media_items':10,
     'distinct_articles':0,'id_namespace':'source_record_id','license':'CC-BY-4.0'}],
    'cross_dataset_article_matches':0,'join_policy':'no_unir_fuentes_distintas_por_article_id'}
(OUT/'source_registry.json').write_text(json.dumps(registro_fuentes,ensure_ascii=False,indent=2),encoding='utf-8')
print('INTEGRACIÓN DE INVENTARIOS: 32 referencias de imágenes (24 noticias) + 10 WAV reales autorizados')
print('RELACIONES EDITORIALES ENTRE DATASETS:',registro_fuentes['cross_dataset_article_matches'])

## 7 · Cargar las dos modalidades en un catálogo verificable

La carga (**L**) no consiste en guardar un archivo y olvidarlo. Una fila describe el activo, su modalidad, la procedencia, su huella, sus características y la ruta que permite reabrirlo.

**HAZ ESTO AHORA.** Escribe JSONL y Parquet, vuelve a leer ambos, y contrasta con la misma tabla. No incluiremos \`3530042\` como identificador de estas muestras: fueron creadas por el curso y no proceden de esa noticia.

In [ ]:
# L — esquema común; dos perfiles de imagen y un audio original del curso.
ARTICULO_DEMO='curso-demo-s10' # No es un artículo de un medio
rows=[]
for lado in (160,320):
    ruta=DER/f'documento_{lado}.png';features=imagen_caracteristicas(ruta)
    rows.append({'article_id':ARTICULO_DEMO,'modality':'image','source_kind':'generated_for_course',
       'asset_id':sha256(imagen_original),'original_path':str(imagen_original.relative_to(RAIZ)),
       'derived_path':str(ruta.relative_to(RAIZ)),'sha256':sha256(ruta),
       'profile':f'vista{lado}','feature_version':'s10-v1',
       'metric_name':'mean_luminance','metric_value':features['mean_luminance'],
       'width':features['width'],'height':features['height'],'duration_seconds':None,'sample_rate_hz':None})
rows.append({'article_id':ARTICULO_DEMO,'modality':'audio','source_kind':'generated_for_course',
   'asset_id':sha256(audio_original),'original_path':str(audio_original.relative_to(RAIZ)),
   'derived_path':str(audio_original.relative_to(RAIZ)),'sha256':sha256(audio_original),
   'profile':'original_pcm16','feature_version':'s10-v1',
   'metric_name':'global_rms','metric_value':round(float(np.sqrt(np.mean(senal**2))),5),
   'width':None,'height':None,'duration_seconds':len(senal)/SR,'sample_rate_hz':SR})
catalogo=pd.DataFrame(rows)
catalogo.to_json(OUT/'assets.jsonl',orient='records',lines=True,force_ascii=False)
catalogo.to_parquet(OUT/'assets.parquet',index=False)
reabierto=pd.read_parquet(OUT/'assets.parquet')
jsonl=pd.read_json(OUT/'assets.jsonl',orient='records',lines=True,convert_dates=False,dtype={'article_id':str})
assert reabierto.to_json(orient='records')==jsonl.to_json(orient='records')
assert reabierto.groupby('modality').size().to_dict()=={'audio':1,'image':2}
display(reabierto[['article_id','modality','profile','metric_name','metric_value','sha256']])
print('L · CATÁLOGO REABIERTO:',len(reabierto),'filas')

## 8 · Tu pregunta, sobre tu resultado

Elige qué modalidad consultar. **No redactes una justificación:** toma una decisión verificable con tu propia salida y contrasta el archivo que se abre.

Si necesitas recuperar el ejemplo, vuelve a elegir \`image\`. No confundas las tres filas del catálogo con tres fuentes independientes: son dos derivaciones de imagen y un audio.

In [ ]:
# Ejercicio de consulta: cambiar SOLO la modalidad.
MODALIDAD='image' #@param ['image','audio']
if MODALIDAD not in ('image','audio'):raise ValueError('Elige una modalidad del catálogo')
resultado=reabierto[reabierto.modality.eq(MODALIDAD)]
assert len(resultado)>0
display(resultado[['modality','profile','metric_name','metric_value','derived_path']])
for _,r in resultado.iterrows():
    p=RAIZ/r.derived_path
    assert sha256(p)==r.sha256
    if r.modality=='image':display(Image.open(p))
    else:
        with wave.open(str(p),'rb') as wf:
            a=np.frombuffer(wf.readframes(wf.getnframes()),dtype='<i2').astype(np.float32)/32768
            display(Audio(a,rate=wf.getframerate()))
print('CONSULTA PERSONAL:',MODALIDAD,'· archivos reabiertos:',len(resultado))

**Cómo se lee:** la consulta selecciona los archivos por modalidad y verifica cada huella antes de abrirlos. **Qué nos dice:** es un catálogo utilizable, no una colección de rutas rotas. **Qué NO permite concluir:** asociación semántica con noticias reales, calidad de OCR o ASR, ni desempeño de un clúster. **Error común:** confundir integridad de bytes con veracidad del contenido.

## 9 · ¿Dónde aparece Big Data?

Un ejemplo pequeño nos permite inspeccionar cada transformación; el diseño debe contemplar muchos activos, pero **no vamos a inventar una prueba de rendimiento distribuido**. Estimar memoria potencial permite decidir qué guardar como binario y qué guardar como características.

Para un millón de imágenes de las mismas dimensiones, compáralas como RGB sin comprimir y como PNG derivado real. Para audio PCM, relaciona segundos, muestras, bytes por muestra y canales. Estas son proyecciones hipotéticas basadas en un ejemplo, **no estadísticas sobre noticias reales**.

In [ ]:
# Proyección matemática, no benchmark ni descarga masiva.
N_ARCHIVOS=1000000
medidas=imagen_caracteristicas(imagen_original)
rgb_bytes=medidas['width']*medidas['height']*3
imagen_bytes=imagen_original.stat().st_size
audio_pcm_bytes=len(senal)*2
print('UN ORIGINAL · imagen sin comprimir:',rgb_bytes,'bytes; PNG en disco:',imagen_bytes,'bytes')
print('UN ORIGINAL · audio WAV PCM16 (sin cabecera):',audio_pcm_bytes,'bytes')
print('1 MILLÓN HIPOTÉTICO de imágenes RGB:',round(rgb_bytes*N_ARCHIVOS/1e9,2),'GB decimales')
print('1 MILLÓN HIPOTÉTICO de audios de',duracion,'s:',round(audio_pcm_bytes*N_ARCHIVOS/1e9,2),'GB decimales')
print('NO SE MIDIERON red, concurrencia, latencia ni almacenamiento distribuido.')
print('DECISIÓN: conservar originales y un índice pequeño de características evita decodificar todo en cada consulta.')

## 9 · Decisión cerrada y verificación final

Elige un perfil de imagen (1=160, 2=320), una región de audio (1=primer tono, 2=silencio, 3=segundo tono) y un límite (1=histograma no es OCR, 2=espectro no es transcripción, 3=mis muestras no representan noticias reales).

No hay nota en este cuaderno. El objetivo es dejar un producto verificable que puedas defender ante tus compañeros.

**HAZ ESTO AHORA.** Sustituye los ceros de tu decisión; la ejecución rechazará opciones fuera de rango.

In [ ]:
# Selecciones estructuradas, con 0 = pendiente.
PERFIL_IMAGEN=0 #@param [0,1,2] {type:"raw"}
REGION_AUDIO=0 #@param [0,1,2,3] {type:"raw"}
LIMITE=0 #@param [0,1,2,3] {type:"raw"}
for v,validos in [(PERFIL_IMAGEN,(0,1,2)),(REGION_AUDIO,(0,1,2,3)),(LIMITE,(0,1,2,3))]:
    if type(v) is not int or v not in validos:raise ValueError('Código de decisión inválido')
decision={'profile':PERFIL_IMAGEN,'audio_region':REGION_AUDIO,'limit':LIMITE,
          'complete':bool(PERFIL_IMAGEN and REGION_AUDIO and LIMITE),'graded':False}
(CHECKS/'decision.json').write_text(json.dumps(decision,indent=2,ensure_ascii=False),encoding='utf-8')
print('DECISIÓN INDIVIDUAL:',decision)
if not decision['complete']:print('PENDIENTE: selecciona tres códigos distintos de cero para completar tu reflexión.')

In [ ]:
# Manifiesto técnico: huellas y relectura del catálogo/archivos.
from datetime import datetime,timezone
def comprobar(paquete):
    p=Path(paquete);m=json.loads((p/'manifest_s10.json').read_text(encoding='utf-8'))
    errors=[]
    for ruta,esperado in m['files'].items():
        a=p/ruta
        if not a.is_file() or sha256(a)!=esperado:errors.append('hash:'+ruta)
    tabla=pd.read_parquet(p/'outputs/assets.parquet')
    if len(tabla)!=m['rows']:errors.append('rows')
    for _,fila in tabla.iterrows():
        ruta=p/fila['derived_path']
        if not ruta.is_file() or sha256(ruta)!=fila['sha256']:errors.append('asset:'+str(ruta))
    return {'ok':not errors,'errors':errors,'rows':len(tabla)}
filelist=[*RAW.glob('*'),*DER.glob('*'),OUT/'assets.jsonl',OUT/'assets.parquet',CHECKS/'decision.json']
manifest={'schema':'bigdata-s10-multimodal-v1','created_at_utc':datetime.now(timezone.utc).isoformat(),
          'source_kind':'generated_for_course','article_id':'curso-demo-s10',
          'rows':len(reabierto),
          'files':{p.relative_to(RAIZ).as_posix():sha256(p) for p in filelist}}
(RAIZ/'manifest_s10.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
informe=comprobar(RAIZ)
assert informe['ok'],informe
print('PAQUETE VERIFICADO:',informe,'· carpeta:',RAIZ)

In [ ]:
#@title Opcional: conservar la carpeta íntegra en TU Google Drive (sin hacerlo obligatorio para VS Code)
CONSERVAR_EN_DRIVE=False #@param {type:"boolean"}
if CONSERVAR_EN_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        raise RuntimeError('Esta opción es solo para Colab; en VS Code conserva RAIZ localmente')
    drive.mount('/content/drive')
    destino=Path('/content/drive/MyDrive/BigData2026/S10/multimodal')
    destino.parent.mkdir(parents=True,exist_ok=True)
    shutil.copytree(RAIZ,destino,dirs_exist_ok=True)
    verificado=comprobar(destino)
    assert verificado['ok'],verificado
    print('CONSERVADO EN DRIVE:',destino,'· COPIA REABIERTA:',verificado['ok'])
else:
    print('CONSERVACIÓN LOCAL:',RAIZ,'· para Colab activa la copia a Drive antes de cerrar el runtime')

## Para llevar · hoja de consulta

| Necesidad | Operación |
|---|---|
| Conservar original | \`shutil.copyfile()\` / \`Image.save()\` / \`wave.open()\` + SHA-256 |
| Leer píxeles | \`PIL.Image.open()\` → \`np.asarray()\` |
| Reducir imagen | \`thumbnail((lado,lado))\` |
| Medir sonido | \`np.sqrt(np.mean(x**2))\` |
| Frecuencias | \`np.fft.rfft()\` y \`np.fft.rfftfreq()\` |
| Consultar catálogo | \`pd.read_parquet()\` + filtro por modalidad |
| Reabrir y verificar | \`sha256(path)\` + manifiesto |
| Transcribir voz | Modelo ASR autorizado, texto de referencia y evaluación (extensión) |

**Puente a S11:** conserva \`work_s10_multimodal/\`, porque ya tiene el contrato de un catálogo multimedia. La siguiente sesión puede añadir más registros de audio, imágenes reales con permisos, otros formatos y una ejecución por lotes con métricas de escala.

**Idea final:** E obtiene y acredita el origen; T produce características verificables; L almacena una representación que otro proceso puede reabrir. Una característica técnica no equivale a interpretación humana.